# Hazard Combination Exposure

Computes which combination of hazard topics each child is exposed to, per country.

**Method:** Bitmask encoding — each of 8 hazard topics is assigned a power-of-2 value.
Pixel-level sum yields a unique `combination_id` for every subset of active topics.
Child population is grouped by `combination_id` within each country chunk,
then aggregated and decoded in Python.

**Phase 1 (GEE):** Build binary masks → combination_id image → `reduceRegions` with grouped sum → export chunk-level CSV  
**Phase 2 (Python):** Parse groups → aggregate chunks by country → decode bitmask → label Single/Double/Triple/Multiple

In [ ]:
import ee
import pandas as pd
import json
import re
import numpy as np

ee.Authenticate()
ee.Initialize(project='unicef-ccri')

In [ ]:
# =========================================================
# 1. LOAD ASSETS
# =========================================================

# Child population (WorldPop constrained U18, 100m)
org_childpop = ee.ImageCollection('projects/unicef-ccri/assets/population/worldpop_T_U18_2025_CN_100m')
childpop = org_childpop.mosaic().setDefaultProjection(org_childpop.first().projection())
pop_res = org_childpop.first().projection().nominalScale()

# Chunked ADM0 boundaries (~500km chunks for large countries)
chunked_boundaries = ee.FeatureCollection('projects/unicef-ccri/assets/global_boundary/adm0_chunked_500km_shp')

print('pop_res:', pop_res.getInfo(), 'm')
print('chunked_boundaries size:', chunked_boundaries.size().getInfo())

In [ ]:
# =========================================================
# 2. HAZARD DEFINITIONS
# =========================================================

hazards = [
    {'id': 'projects/unicef-ccri/assets/hazards/river_flood_r100',
     'threshold': 0.01,   'name': 'river_flood_100yr_jrc_2024',          'type': 'Collection'},
    {'id': 'projects/unicef-ccri/assets/hazards/coastal_flood_r100',
     'threshold': 0,      'name': 'coastal_flood_100yr_jrc_2024',         'type': 'Collection'},
    {'id': 'projects/unicef-ccri/assets/hazards/storm_giri_rp100',
     'threshold': 17.5,   'name': 'tropical_storm_100yr_giri_2024',       'type': 'Collection'},
    {'id': 'projects/unicef-ccri/assets/hazards/ASI_return_level_100yr',
     'threshold': 30,     'name': 'agricultural_drought_fao_1984-2023',   'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/droughts/spei12_TerraClimate_1958-2025',
     'band': 'b2', 'threshold': 0.0650162152126539,
     'name': 'drought_spei_terraclimate_1958-2025',                       'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/droughts/spi12_TerraClimate_1958-2025',
     'band': 'b2', 'threshold': 0.0912838950900999,
     'name': 'drought_spi_terraclimate_1958-2025',                        'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/heatwave_frequency_return_level_100yr',
     'threshold': 16.02,  'name': 'heatwave_frequency_ecmwf_2014-2024',   'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/heatwave_duration_return_level_100yr',
     'threshold': 94.01,  'name': 'heatwave_duration_ecmwf_2014-2024',    'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/heatwave_severity_return_level_100yr',
     'threshold': 3.66,   'name': 'heatwave_severity_ecmwf_2014-2024',    'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/high_temp_degree_days_return_level_100yr',
     'threshold': 35,     'name': 'extreme_heat_ecmwf_2014-2024',         'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/FIRMS_FRP_90th_percentile',
     'threshold': 37.89,  'name': 'fire_FRP_nasa_2001-2024',              'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/FIRMS_count_90th_percentile',
     'threshold': 4.91,   'name': 'fire_frequency_nasa_2001-2023',        'type': 'Image'},
    {'id': 'projects/unicef-ccri/assets/hazards/sand_dust_storm_annual',
     'threshold': 0,      'name': 'sand_dust_storm_unccd_2024',           'type': 'Image',
     'no_data': -999000000},
]

In [ ]:
# =========================================================
# 3. HAZARD TOPICS
# =========================================================
# Order here determines bit assignment: topic at index i → bit value 2^i.
# This order must be preserved for Phase 2 decoding.

hazard_topics = {
    'River Flood':         ['river_flood_100yr_jrc_2024'],
    'Coastal Flood':       ['coastal_flood_100yr_jrc_2024'],
    'Tropical Storm':      ['tropical_storm_100yr_giri_2024'],
    'Drought':             ['agricultural_drought_fao_1984-2023',
                            'drought_spei_terraclimate_1958-2025',
                            'drought_spi_terraclimate_1958-2025'],
    'Heatwave':            ['heatwave_frequency_ecmwf_2014-2024',
                            'heatwave_duration_ecmwf_2014-2024',
                            'heatwave_severity_ecmwf_2014-2024'],
    'Extreme Heat':        ['extreme_heat_ecmwf_2014-2024'],
    'Fire':                ['fire_FRP_nasa_2001-2024',
                            'fire_frequency_nasa_2001-2023'],
    'Sand and Dust Storm': ['sand_dust_storm_unccd_2024'],
}

# Stable ordered list for bit indexing
topic_names = list(hazard_topics.keys())
print('Topics and bit values:')
for i, t in enumerate(topic_names):
    print(f'  bit {i} (value {2**i:3d}): {t}')

In [ ]:
# =========================================================
# 4. BINARY MASK GENERATOR
# =========================================================

def get_binary_mask(h):
    layer = (
        ee.ImageCollection(h['id']).mosaic()
        if h['type'] == 'Collection'
        else ee.Image(h['id'])
    )
    if 'band' in h:
        layer = layer.select(h['band'])

    if 'no_data' in h:
        layer = layer.updateMask(layer.neq(h['no_data']))

    th = h['threshold']

    # Agricultural drought: valid range 0–100
    if 'agricultural_drought' in h['name']:
        layer = layer.updateMask(layer.gte(0).And(layer.lte(100)))
        return layer.gt(th)

    # TerraClimate SPI/SPEI: probability bands — exposed if >= threshold
    if 'spei' in h['name'] or 'spi' in h['name']:
        return layer.gte(th)

    # Default: exposed if > threshold, inherit input mask
    return layer.gt(th).updateMask(layer.mask())

In [ ]:
# =========================================================
# 5. BUILD BINARY MASKS AND TOPIC UNION MASKS
# =========================================================

exposure_by_name = {h['name']: get_binary_mask(h) for h in hazards}

topic_masks = {}
for topic, names in hazard_topics.items():
    masks = [exposure_by_name[name].unmask(0) for name in names]
    union = masks[0]
    for m in masks[1:]:
        union = union.Or(m)
    topic_masks[topic] = union

print('Topic masks built:', list(topic_masks.keys()))

In [ ]:
# =========================================================
# 6. BUILD COMBINATION_ID IMAGE AND ANALYSIS IMAGE
# =========================================================
# Each topic at index i contributes 2^i to the pixel value.
# Pixel sum = unique bitmask encoding the active topic set.

combination_id = ee.Image.constant(0)
for i, topic in enumerate(topic_names):
    bit_value = int(2 ** i)
    contribution = (ee.Image.constant(bit_value)
                    .updateMask(topic_masks[topic])
                    .unmask(0))
    combination_id = combination_id.add(contribution)

# Mask pixels with no hazard exposure (combination_id = 0)
combination_id = (combination_id
                  .updateMask(combination_id.neq(0))
                  .toInt32()
                  .rename('combination_id'))

# Stack: band 0 = child_population, band 1 = combination_id
# groupField=1 in the reducer references band 1 (combination_id)
analysis_image = ee.Image.cat([
    childpop.rename('child_population'),
    combination_id
])

# Restrict to pixels with non-zero child population
analysis_image = analysis_image.updateMask(childpop.gt(0))

print('Analysis image bands:', analysis_image.bandNames().getInfo())

## Phase 1 — GEE Computation and Export

`reduceRegions` with a grouped `sum` reducer groups child population by `combination_id`
within each boundary chunk. Large countries produce multiple chunk features; Phase 2
aggregates them by `ucode`.

In [ ]:
# =========================================================
# 7. RUN REDUCE REGIONS AND EXPORT
# =========================================================

# groupField=1 references band index 1 (combination_id)
grouped_reducer = ee.Reducer.sum().group(groupField=1, groupName='combination_id')

chunk_stats = analysis_image.reduceRegions(
    collection=chunked_boundaries,
    reducer=grouped_reducer,
    scale=pop_res,
    tileScale=4
)

# Keep only the properties needed for post-processing
def keep_properties(feature):
    return ee.Feature(None, {
        'ucode': feature.get('ucode'),
        'ISO3': feature.get('ISO3'),
        'groups': feature.get('groups')
    })

export_collection = chunk_stats.map(keep_properties)

task = ee.batch.Export.table.toDrive(
    collection=export_collection,
    description='HazardCombination_chunks',
    folder='GEE_Exports',
    fileFormat='CSV'
)
task.start()
print(f'Export task started — ID: {task.id}')
print('Monitor at: https://code.earthengine.google.com/tasks')

## Phase 2 — Post-Processing

After the export completes, run the cells below to:
1. Parse the `groups` column (GEE serializes list properties as `{key=value}` strings)
2. Aggregate chunk rows for the same country by summing population per `combination_id`
3. Decode each `combination_id` back to topic names using the same bitmask order
4. Label combinations as Single / Double / Triple / Multiple

Set `INPUT_CSV` to the path of the exported file before running.

In [ ]:
INPUT_CSV  = '../../data/misc/HazardCombination_chunks.csv'
OUTPUT_CSV = '../../data/misc/HazardCombination_final.csv'

df_raw = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df_raw)} chunk rows')
print(df_raw.head(3))

In [ ]:
# =========================================================
# 9. FLATTEN GROUPS AND AGGREGATE BY COUNTRY
# =========================================================

def parse_gee_groups(s):
    """Parse GEE list-of-dicts property from CSV string.
    GEE serializes as: [{combination_id=41, sum=1200000.0}, ...]
    """
    if pd.isna(s) or str(s).strip() in ('[]', '', 'None', 'null'):
        return []
    # Convert {key=value} → {"key": value} for JSON parsing
    s = re.sub(r'(\w+)=', r'"\1":', str(s).strip())
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return []

# Flatten: one row per (chunk × combination_id)
records = []
for _, row in df_raw.iterrows():
    for g in parse_gee_groups(row['groups']):
        records.append({
            'ucode':          row['ucode'],
            'ISO3':           row.get('ISO3'),
            'combination_id': int(g['combination_id']),
            'population':     float(g['sum'])
        })

df_flat = pd.DataFrame(records)
print(f'Flat rows before aggregation: {len(df_flat)}')

# Aggregate chunks: sum population by (ucode, ISO3, combination_id)
df_agg = (
    df_flat
    .groupby(['ucode', 'ISO3', 'combination_id'], dropna=False)['population']
    .sum()
    .reset_index()
)
print(f'Rows after aggregation: {len(df_agg)}')
print(f'Countries: {df_agg["ucode"].nunique()}')

In [ ]:
# =========================================================
# 10. DECODE BITMASK AND LABEL HAZARD LEVEL
# =========================================================
# topic_names order must match the order used in Phase 1 (cell 3).

def decode_combination(combo_id):
    return [topic_names[i] for i in range(len(topic_names)) if combo_id & (1 << i)]

def label_level(count):
    return {1: 'Single', 2: 'Double', 3: 'Triple'}.get(count, 'Multiple')

df_agg['topics']      = df_agg['combination_id'].apply(decode_combination)
df_agg['combination'] = df_agg['topics'].apply(' + '.join)
df_agg['hazardCount'] = df_agg['topics'].apply(len)
df_agg['hazardLevel'] = df_agg['hazardCount'].apply(label_level)

df_agg = df_agg.drop(columns='topics')

print(df_agg[['ISO3', 'combination', 'hazardLevel', 'population']].head(10))

In [ ]:
# =========================================================
# 11. SORT AND SAVE
# =========================================================

df_final = df_agg.sort_values(
    ['hazardCount', 'population'],
    ascending=[False, False]
).reset_index(drop=True)

df_final.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df_final)} rows to {OUTPUT_CSV}')
print(f"\nHazard level distribution:")
print(df_final['hazardLevel'].value_counts())
print(f"\nTop combinations by exposed children:")
print(
    df_final.groupby('combination')['population']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)